In [ ]:
!pip install -q -U keras-hub
!pip install  -q -U keras
!pip install -q -U keras-nlp

In [ ]:
import pandas as pd
import pyarrow as pa
import keras
import keras_hub
import keras
import keras_nlp
from keras_nlp.samplers import TopKSampler
from time import time
import csv

In [ ]:
import pandas as pd

splits = {'train': 'data/train-00000-of-00001.parquet', 'test': 'data/test-00000-of-00001.parquet'}
df = pd.read_parquet("hf://datasets/Thimira/sinhala-llm-dataset-llama-prompt-format/" + splits["train"])

In [ ]:
def split_text(row):
    prompt = row.split("[/INST]")[0].replace("<s>[INST]", "").strip()
    response = row.split("[/INST]")[1].replace("</s>", "").strip()
    return prompt, response


df[["prompt", "response"]] = df["text"].apply(lambda x: pd.Series(split_text(x)))

prompts = df["prompt"].tolist()
responses = df["response"].tolist()

data = {
    "prompts": prompts,
    "responses": responses
}

In [ ]:
import os 

os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow".
# Avoid memory fragmentation on JAX backend.
os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"]="1.00"
# avoid memory fragmentation on JAX backend.
os.environ["JAX_PLATFORMS"] = ""

In [ ]:
gemma_lm = keras_hub.models.Gemma3CausalLM.from_preset("/kaggle/input/gemma3/keras/gemma3_instruct_270m/4",dtype="int8")
gemma_lm.summary()

In [ ]:
# template = "Instruction:\n{instruction}\n\nResponse:\n{response}"

# prompt = template.format(
#     instruction="What should I do on a trip to Europe?",
#     response="",
# )
# sampler = keras_hub.samplers.TopKSampler(k=5, seed=2)
# gemma_lm.compile(sampler=sampler)
# print(gemma_lm.generate(prompt, max_length=256))

In [ ]:
# Enable LoRA for the model and set the LoRA rank to 5.
gemma_lm.backbone.enable_lora(rank=5)

In [ ]:
gemma_lm.summary()

In [ ]:
gemma_lm.fit(data, epochs=1, batch_size=1)